# Modul A · Kapitel 2 · Teil 4 — GPT trainieren

**Lernziel:** Du kannst Trainingsdaten als Batches bereitstellen, ein Sprachmodell mit
Backpropagation trainieren und anschließend autoregressiv Text erzeugen.

Der Ablauf in diesem Notebook:

`Text → Batch → Logits → Cross-Entropy Loss → Backpropagation → neue Parameter → Textgenerierung`

Es gibt **3 Challenges**. Die Cross-Entropy ist bewusst keine Challenge: Der Code wird gezeigt
und direkt erklärt.

## 0 · Setup

▶️ Führe die nächsten drei Zellen aus. Sie laden PyTorch, den Faust-Korpus, den
Zeichen-Tokenizer und das vollständige `MiniGPT` aus den vorigen Teilen.

In [ ]:
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.nn import functional as F

# Einheitliche Farben für alle Diagramme in diesem Notebook
BLAU, ORANGE, TEAL, GRAU = "#2563eb", "#e8590c", "#0d9488", "#6b7280"

plt.rcParams.update({
    "figure.figsize": (8, 5),
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": GRAU,
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": "#e5e7eb",
    "grid.linewidth": 0.8,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "font.size": 11,
})
torch.manual_seed(1337)

# Wenn eine Grafikkarte da ist, benutzen wir sie — das Training wird dadurch rund dreimal
# schneller. Sonst läuft alles auf der CPU, nur eben gemütlicher.
if torch.cuda.is_available():
    GERAET = "cuda"                                  # NVIDIA-Grafikkarte, z. B. in Colab
elif torch.backends.mps.is_available():
    GERAET = "mps"                                   # Apple Silicon
else:
    GERAET = "cpu"

print(f"PyTorch {torch.__version__}")
print(f"Gerät:   {GERAET}")
print("Setup fertig ✔")

In [ ]:
# ▶️ Der Text, der Zeichen-Tokenizer und die Einstellungen des Modells
def lade_faust():
    """Lädt Goethes Faust I und II — einmalig von Project Gutenberg, danach aus `faust.txt`."""
    import re
    import urllib.request

    pfad = Path("faust.txt")
    if pfad.exists():
        print(f"Gelesen: {pfad}")
        return pfad.read_text(encoding="utf-8")

    print("faust.txt nicht gefunden — lade von Project Gutenberg (rund 0,5 MB) …")

    def teil(nummer, ab):
        """Holt ein Buch und schneidet Vorspann, Nachspann und Inhaltsverzeichnis weg."""
        adresse = f"https://www.gutenberg.org/cache/epub/{nummer}/pg{nummer}.txt"
        roh = urllib.request.urlopen(adresse).read().decode("utf-8").replace("\r\n", "\n")
        roh = roh[:roh.find("*** END OF THE PROJECT GUTENBERG")]
        roh = "\n".join(z[2:] if z.startswith("  ") else z for z in roh.split("\n"))
        return roh[roh.find(ab):]

    erster = teil(2229, "Zueignung\n\n\nIhr naht")
    zweiter = teil(2230, "1.  Akt--Anmutige Gegend")
    # Faust II schreibt die Sprechernamen mit Doppelpunkt, Faust I mit Punkt — wir vereinheitlichen
    zweiter = re.sub(r"(?m)^([A-ZÄÖÜ][A-ZÄÖÜ \-]*(?:\(.*?\))?):$", r"\1.", zweiter)

    text = re.sub(r"\n{3,}", "\n\n", erster.strip() + "\n\n" + zweiter.strip()) + "\n"
    Path("faust.txt").write_text(text, encoding="utf-8")
    print("Gespeichert als: faust.txt")
    return text


text = lade_faust()

zeichen = sorted(set(text))
vokabular_groesse = len(zeichen)
zeichen_zu_zahl = {z: i for i, z in enumerate(zeichen)}
zahl_zu_zeichen = {i: z for i, z in enumerate(zeichen)}


def kodiere(s):
    """Text → Liste von Zahlen."""
    return [zeichen_zu_zahl[z] for z in s]


def dekodiere(zahlen):
    """Liste von Zahlen → Text."""
    return "".join(zahl_zu_zeichen[i] for i in zahlen)


KONTEXT = 96      # wie viele Zeichen das Modell zurückschaut (block size)
BATCH = 32        # wie viele Textstücke gleichzeitig gerechnet werden
N_EMBD = 96       # Länge der Vektoren, mit denen das Modell intern arbeitet
N_HEAD = 4        # Aufmerksamkeitsköpfe je Schicht
N_LAYER = 3       # wie viele Transformer-Blöcke übereinander
DROPOUT = 0.1     # Anteil der Verbindungen, der beim Training zufällig abgeschaltet wird

SCHRITTE = 1500   # Trainingsschritte — genug für einen klar sichtbaren Lernfortschritt
LERNRATE = 1e-3

print(f"{len(text):,} Zeichen".replace(",", ".") + f", {vokabular_groesse} davon verschieden")
print(f"Kontextlänge {KONTEXT}, {N_LAYER} Blöcke à {N_HEAD} Köpfe, {N_EMBD} Dimensionen")

In [ ]:
# ▶️ Das vollständige Modell — Aufmerksamkeit, Feed-Forward, Block, MiniGPT
class Kopf(nn.Module):
    """Ein einzelner Aufmerksamkeitskopf: Query, Key, Value, maskiert und skaliert."""

    def __init__(self, kopf_groesse):
        super().__init__()
        self.query = nn.Linear(N_EMBD, kopf_groesse, bias=False)
        self.key = nn.Linear(N_EMBD, kopf_groesse, bias=False)
        self.value = nn.Linear(N_EMBD, kopf_groesse, bias=False)
        self.dropout = nn.Dropout(DROPOUT)
        self.register_buffer("maske", torch.tril(torch.ones(KONTEXT, KONTEXT)))

    def forward(self, x):
        B, T, C = x.shape
        q = self.query(x)
        k = self.key(x)
        v = self.value(x)
        gewichte = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5
        gewichte = gewichte.masked_fill(self.maske[:T, :T] == 0, float("-inf"))
        gewichte = F.softmax(gewichte, dim=-1)
        gewichte = self.dropout(gewichte)
        return gewichte @ v


class MehrereKoepfe(nn.Module):
    """N_HEAD Köpfe parallel, danach eine lineare Schicht, die sie zusammenmischt."""

    def __init__(self, anzahl, kopf_groesse):
        super().__init__()
        self.koepfe = nn.ModuleList([Kopf(kopf_groesse) for _ in range(anzahl)])
        self.projektion = nn.Linear(anzahl * kopf_groesse, N_EMBD)
        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, x):
        zusammen = torch.cat([kopf(x) for kopf in self.koepfe], dim=-1)
        return self.dropout(self.projektion(zusammen))


class FeedForward(nn.Module):
    """Zwei lineare Schichten mit ReLU dazwischen — je Position einzeln angewendet."""

    def __init__(self):
        super().__init__()
        self.netz = nn.Sequential(
            nn.Linear(N_EMBD, 4 * N_EMBD),
            nn.ReLU(),
            nn.Linear(4 * N_EMBD, N_EMBD),
            nn.Dropout(DROPOUT),
        )

    def forward(self, x):
        return self.netz(x)


class Block(nn.Module):
    """Aufmerksamkeit, dann Feed-Forward — beides mit Layer Norm davor und Skip Connection."""

    def __init__(self):
        super().__init__()
        self.aufmerksamkeit = MehrereKoepfe(N_HEAD, N_EMBD // N_HEAD)
        self.feedforward = FeedForward()
        self.norm1 = nn.LayerNorm(N_EMBD)
        self.norm2 = nn.LayerNorm(N_EMBD)

    def forward(self, x):
        x = x + self.aufmerksamkeit(self.norm1(x))
        x = x + self.feedforward(self.norm2(x))
        return x


class MiniGPT(nn.Module):
    """Embeddings → N_LAYER Blöcke → Layer Norm → Ausgabeschicht auf 87 Logits."""

    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(vokabular_groesse, N_EMBD)
        self.positions_embedding = nn.Embedding(KONTEXT, N_EMBD)
        self.bloecke = nn.Sequential(*[Block() for _ in range(N_LAYER)])
        self.norm_ende = nn.LayerNorm(N_EMBD)
        self.ausgabe = nn.Linear(N_EMBD, vokabular_groesse)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        x = self.token_embedding(idx) + self.positions_embedding(torch.arange(T, device=idx.device))
        x = self.bloecke(x)
        x = self.norm_ende(x)
        logits = self.ausgabe(x)                          # (B, T, 87)

        if targets is None:
            return logits, None

        loss = F.cross_entropy(logits.view(B * T, vokabular_groesse), targets.view(B * T))
        return logits, loss


torch.manual_seed(1337)
modell = MiniGPT().to(GERAET)

anzahl_parameter = sum(p.numel() for p in modell.parameters())
print(f"MiniGPT mit {anzahl_parameter:,} Parametern — noch untrainiert".replace(",", "."))
print()
for name, teil in [("Token-Embedding", modell.token_embedding),
                   ("Positions-Embedding", modell.positions_embedding),
                   ("Blöcke", modell.bloecke),
                   ("Abschluss-LayerNorm", modell.norm_ende),
                   ("Ausgabeschicht", modell.ausgabe)]:
    n = sum(p.numel() for p in teil.parameters())
    anteil = f"{n / anzahl_parameter:.1%}".replace(".", ",")
    print(f"  {name:<22}{n:>9,}".replace(",", ".") + f"   {anteil:>6}")

## 1 · Trainingsdaten und Batches

Das Modell soll an jeder Position das **nächste Zeichen** vorhersagen. Deshalb ist `y` derselbe
Text wie `x`, aber um eine Position nach links verschoben.

Wir verwenden 90 % des Textes als **Training Set** und 10 % als **Validation Set**. Der
Validation Loss zeigt später, wie gut das Modell auf ungesehenem Text funktioniert.

In [ ]:
daten = torch.tensor(kodiere(text), dtype=torch.long)
grenze = int(0.9 * len(daten))
train_daten = daten[:grenze]
val_daten = daten[grenze:]

beispiel = train_daten[:9]
print("x:", repr(dekodiere(beispiel[:-1].tolist())))
print("y:", repr(dekodiere(beispiel[1:].tolist())))
print(f"Training: {len(train_daten):,} · Validation: {len(val_daten):,}".replace(",", "."))

### 🛠️ Challenge 1 — Einen Batch erzeugen

Implementiere `get_batch(split)`. Ziehe `BATCH` zufällige Startpositionen und erzeuge:

- `x` mit der Shape `(BATCH, KONTEXT)`
- `y` mit derselben Shape, um eine Position verschoben

Ein Batch verarbeitet viele Textausschnitte parallel. Zusätzlich liefert jede der `KONTEXT`
Positionen ein eigenes Trainingssignal.

In [ ]:
def get_batch(split):
    source = train_daten if split == "train" else val_daten
    starts = torch.randint(len(source) - KONTEXT - 1, (BATCH,))

    # TODO: BATCH Ausschnitte stapeln
    x = ...
    y = ...

    return x.to(GERAET), y.to(GERAET)

In [ ]:
x, y = get_batch("train")
assert x.shape == y.shape == (BATCH, KONTEXT)
assert torch.equal(x[:, 1:], y[:, :-1])
print("✅ Batch korrekt:", tuple(x.shape))

<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
def get_batch(split):
    source = train_daten if split == "train" else val_daten
    starts = torch.randint(len(source) - KONTEXT - 1, (BATCH,))
    x = torch.stack([source[i:i + KONTEXT] for i in starts])
    y = torch.stack([source[i + 1:i + KONTEXT + 1] for i in starts])
    return x.to(GERAET), y.to(GERAET)
```

</details>

## 2 · Cross-Entropy Loss

Für jede Position erzeugt das Modell einen **Logit** pro Zeichen im Vokabular. Logits sind noch
keine Wahrscheinlichkeiten. Die Cross-Entropy kombiniert intern `log_softmax` und den Negative
Log-Likelihood Loss:

$$\text{Cross-Entropy Loss} = -\frac{1}{N}\sum_i \log p_i(\text{korrektes Zeichen})$$

Je mehr Wahrscheinlichkeit das Modell dem korrekten Zeichen gibt, desto kleiner wird der Loss.
Für die Berechnung erwartet PyTorch eine Matrix `(N, Vokabular)` und `N` Target-IDs. Deshalb
fassen wir Batch- und Zeitdimension zusammen. `F.cross_entropy` ist dabei numerisch stabiler als
eine selbst geschriebene Folge aus `softmax` und `log`.

In [ ]:
x, y = get_batch("train")
logits, _ = modell(x)

B, T, V = logits.shape
loss = F.cross_entropy(
    logits.reshape(B * T, V),
    y.reshape(B * T),
)

print("Logits:", tuple(logits.shape))
print(f"Cross-Entropy Loss: {loss.item():.3f}")
print(f"Referenz für zufälliges Raten: ln({V}) = {np.log(V):.3f}")

Im `forward`-Pass von `MiniGPT` steht dieselbe Berechnung bereits. Darum liefert
`modell(x, y)` direkt `(logits, loss)`. Ein untrainiertes Modell startet ungefähr beim Loss
`ln(Vokabulargröße)`; während des Trainings soll vor allem der **Validation Loss** sinken.

## 3 · Training

Ein Trainingsschritt besteht aus vier Operationen:

1. Forward Pass und Loss berechnen
2. alte Gradienten mit `zero_grad()` löschen
3. mit `backward()` die Gradienten berechnen
4. mit `step()` die Parameter aktualisieren

Wir verwenden **AdamW** als Optimizer.

In [ ]:
optimizer = torch.optim.AdamW(modell.parameters(), lr=LERNRATE)


@torch.no_grad()
def estimate_loss(num_batches=20):
    modell.eval()
    result = {}
    for split in ("train", "val"):
        losses = []
        for _ in range(num_batches):
            x, y = get_batch(split)
            _, loss = modell(x, y)
            losses.append(loss.item())
        result[split] = float(np.mean(losses))
    modell.train()
    return result

### 🛠️ Challenge 2 — Einen Trainingsschritt implementieren

Vervollständige `train_step(x, y)`. Achte besonders auf die Reihenfolge: Gradienten werden vor
`backward()` gelöscht.

In [ ]:
def train_step(x, y):
    # TODO: Forward Pass
    _, loss = ...

    # TODO: Gradienten löschen, Backpropagation, Parameter-Update
    ...
    ...
    ...

    return loss.item()

In [ ]:
x, y = get_batch("train")
before = modell.token_embedding.weight.detach().clone()
batch_loss = train_step(x, y)

assert isinstance(batch_loss, float)
assert not torch.equal(before, modell.token_embedding.weight.detach())
print(f"✅ Parameter wurden aktualisiert · Batch Loss: {batch_loss:.3f}")

<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
def train_step(x, y):
    _, loss = modell(x, y)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    return loss.item()
```

</details>

▶️ Jetzt wiederholen wir den Trainingsschritt. Die Messung verwendet mehrere Batches, damit die
Kurve weniger schwankt. Je nach Hardware dauert die Zelle ungefähr eine bis mehrere Minuten.

In [ ]:
# Reproduzierbarer Neustart nach dem Selbsttest
torch.manual_seed(1337)
modell = MiniGPT().to(GERAET)
optimizer = torch.optim.AdamW(modell.parameters(), lr=LERNRATE)

history = []
start_time = time.time()

for step in range(SCHRITTE + 1):
    if step % 300 == 0:
        losses = estimate_loss()
        history.append((step, losses["train"], losses["val"]))
        print(f"Step {step:>4} · Train Loss {losses['train']:.3f} · "
              f"Validation Loss {losses['val']:.3f}")

    if step < SCHRITTE:
        train_step(*get_batch("train"))

print(f"Fertig nach {time.time() - start_time:.0f} Sekunden auf {GERAET}.")

In [ ]:
steps, train_losses, val_losses = zip(*history)
plt.plot(steps, train_losses, marker="o", label="Train Loss")
plt.plot(steps, val_losses, marker="o", label="Validation Loss")
plt.xlabel("Training Step")
plt.ylabel("Cross-Entropy Loss")
plt.title("Training Curve")
plt.legend()
plt.show()

Die Parameter werden nur mit dem Training Set optimiert. Der Validation Loss wird ausschließlich
gemessen. Steigt er, während der Train Loss weiter fällt, ist das ein Zeichen für **Overfitting**.

## 4 · Autoregressive Textgenerierung

Das trainierte Modell erzeugt Text Zeichen für Zeichen:

1. letzten Kontext an das Modell geben
2. Logits der letzten Position auswählen
3. Logits durch die **Temperature** teilen und Softmax anwenden
4. ein Zeichen sampeln und an den Kontext anhängen

Eine niedrige Temperature macht die Verteilung schärfer und den Text berechenbarer; eine hohe
Temperature erhöht die Vielfalt.

### 🛠️ Challenge 3 — Text generieren

Vervollständige `generate`. `torch.multinomial(probabilities, 1)` sampelt eine Token-ID aus der
Wahrscheinlichkeitsverteilung.

In [ ]:
@torch.no_grad()
def generate(model, idx, max_new_tokens, temperature=1.0):
    model.eval()
    for _ in range(max_new_tokens):
        context = idx[:, -KONTEXT:]

        # TODO: Forward Pass und Logits der letzten Position
        logits, _ = ...
        logits = ...

        # TODO: Wahrscheinlichkeiten und nächstes Zeichen
        probabilities = ...
        next_token = ...

        idx = torch.cat((idx, next_token), dim=1)
    return idx

In [ ]:
prompt = torch.tensor([kodiere("FAUST.\n")], device=GERAET)
generated = generate(modell, prompt, max_new_tokens=600, temperature=0.8)

assert generated.shape == (1, prompt.shape[1] + 600)
print("✅ Generierung funktioniert\n")
print(dekodiere(generated[0].tolist()))

<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
@torch.no_grad()
def generate(model, idx, max_new_tokens, temperature=1.0):
    model.eval()
    for _ in range(max_new_tokens):
        context = idx[:, -KONTEXT:]
        logits, _ = model(context)
        logits = logits[:, -1, :] / temperature
        probabilities = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probabilities, num_samples=1)
        idx = torch.cat((idx, next_token), dim=1)
    return idx
```

</details>

## Nutze jetzt dein selbst trainiertes Mini-GPT

Dein Modell enthält jetzt die Parameter, die es während des Trainings gelernt hat. Passe den
Prompt und die Temperature an und lasse dein eigenes Mini-GPT neuen Text schreiben.

- PROMPT ist der Anfang des Textes.
- MAX_NEW_TOKENS bestimmt die Länge der Fortsetzung.
- Eine niedrige TEMPERATURE erzeugt berechenbareren Text; eine höhere sorgt für mehr Vielfalt.

Der Prompt darf nur Zeichen enthalten, die im Trainingskorpus vorkommen.

In [ ]:
# ▶️ Experimentiere mit deinem selbst trainierten Mini-GPT
PROMPT = "MEPHISTOPHELES.\n"
MAX_NEW_TOKENS = 600
TEMPERATURE = 0.8

unknown_characters = set(PROMPT) - set(zeichen)
assert not unknown_characters, f"Diese Zeichen kennt das Modell nicht: {unknown_characters}"

prompt_tokens = torch.tensor([kodiere(PROMPT)], device=GERAET)
generated_tokens = generate(
    modell,
    prompt_tokens,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
)

print(dekodiere(generated_tokens[0].tolist()))

## Fazit

Du hast den vollständigen Trainingsfluss ausgeführt: Batches erzeugen, Cross-Entropy Loss
berechnen, Gradienten per Backpropagation bestimmen, Parameter mit AdamW aktualisieren und Text
autoregressiv generieren.

Zum Experimentieren: Ändere die `temperature` auf `0.3`, `1.0` oder `1.5`. Beobachte dabei die
Balance zwischen Wiederholung und Vielfalt.